In [6]:
import pandas as pd
import numpy as np

# 1. Load the perfectly unbroken dataset from Notebook 2
# 1. Load the cleaned data
url = "https://github.com/shalomali/ed-arrival-forecasting/raw/main/data/processed/ed_imputed_ready_for_features.csv"

df = pd.read_csv(url)

# Ensure Datetime is the index and properly formatted
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.set_index('Datetime')
df = df.sort_index()

# --- FEATURE SET 1: Calendar & Time Features ---
# Translating time into categorical numbers the model can group by.
df['Hour'] = df.index.hour
df['DayOfWeek'] = df.index.dayofweek # 0 = Monday, 6 = Sunday
df['Month'] = df.index.month
df['Is_Weekend'] = (df.index.dayofweek >= 5).astype(int) # 1 if Sat/Sun, 0 if Weekday

# --- FEATURE SET 2: Lag Features (The Past) ---
# "Shift" the data down to show what happened in the recent past.
# If Lag_1h is high, the model knows we are currently in a surge.
df['Lag_1h'] = df['Arrivals'].shift(1)
df['Lag_2h'] = df['Arrivals'].shift(2)
df['Lag_24h'] = df['Arrivals'].shift(24)   # Exact same hour yesterday
df['Lag_168h'] = df['Arrivals'].shift(168) # Exact same hour last week (24 * 7)

# --- FEATURE SET 3: Rolling Window Features (Broader Trends) ---
# We completely drop the 3-hour window because it acts as a data proxy.
# We shift first, then calculate wider windows (6h, 12h, 24h) to capture true momentum.
df['Rolling_Mean_6h'] = df['Arrivals'].shift(1).rolling(window=6).mean()
df['Rolling_Mean_12h'] = df['Arrivals'].shift(1).rolling(window=12).mean()
df['Rolling_Mean_24h'] = df['Arrivals'].shift(1).rolling(window=24).mean()

# --- CLEANUP ---
# Because we looked exactly 1 week (168 hours) into the past,
# the first 168 rows of our dataset will have 'NaN' for that column.
# Machine learning models will crash on NaNs, so we must drop these initial warmup rows.
df_features = df.dropna().copy()

print(f"Total Rows Before Drop: {len(df)}")
print(f"Total Rows After Dropping Warmup NaNs: {len(df_features)}")
print("-" * 40)

# Preview the final, feature-rich dataset
df_features.head()

Total Rows Before Drop: 17496
Total Rows After Dropping Warmup NaNs: 16186
----------------------------------------


,Date,Arrivals,Hour,DayOfWeek,Month,Is_Weekend,Lag_1h,Lag_2h,Lag_24h,Lag_168h,Rolling_Mean_6h,Rolling_Mean_12h,Rolling_Mean_24h
Datetime,,,,,,,,,,,,,
2014-01-08 00:00:00,2014-01-08,4.0,0,2,1,0,3.0,4.0,1.0,2.0,3.833333,4.000000,3.375000
2014-01-08 01:00:00,2014-01-08,1.0,1,2,1,0,4.0,3.0,1.0,2.0,3.333333,4.250000,3.500000
2014-01-08 02:00:00,2014-01-08,1.0,2,2,1,0,1.0,4.0,2.0,3.0,2.666667,3.833333,3.500000
2014-01-08 04:00:00,2014-01-08,1.0,4,2,1,0,1.0,1.0,1.0,1.0,2.333333,3.416667,3.458333
2014-01-08 05:00:00,2014-01-08,3.0,5,2,1,0,1.0,1.0,1.0,2.0,1.833333,3.333333,3.458333


In [8]:
# Save the final dataset. This is the exact file that goes into our Machine Learning models!
df_features.to_csv('ed_model_ready.csv')